# DSPy ReAct Agents

The `ReAct` module builds agents that reason and act by using tools in an iterative loop.

**What You'll Learn:**
- How ReAct (Reasoning + Acting) works
- Defining tools for agents
- Building a simple agent with `dspy.ReAct`
- Observation/action loops
- When to use ReAct vs simpler modules

## Setup

In [ ]:
import os
import sys
sys.path.append('../../')

import dspy
from utils import setup_default_lm, print_step, print_result, print_error, configure_dspy
from dotenv import load_dotenv

load_dotenv('../../.env')

try:
    lm = setup_default_lm(provider="openai", model="gpt-4o", max_tokens=1000)
    configure_dspy(lm=lm)
    print_result("Language model configured successfully!", "Status")
except Exception as e:
    print_error(f"Failed to configure language model: {e}")
    print("Make sure you have set your OPENAI_API_KEY in the .env file")

## Define Tools

Tools are Python functions with clear docstrings that the agent can call.

In [ ]:
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result.

    Args:
        expression: A mathematical expression to evaluate (e.g., '2 + 3 * 4')
    """
    try:
        allowed_names = {"__builtins__": {}}
        result = eval(expression, allowed_names, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


def word_count(text: str) -> str:
    """Count the number of words in the given text.

    Args:
        text: The text to count words in
    """
    return f"{len(text.split())} words"


def lookup_capital(country: str) -> str:
    """Look up the capital city of a country.

    Args:
        country: The name of the country
    """
    capitals = {
        "france": "Paris", "japan": "Tokyo", "brazil": "Brasilia",
        "australia": "Canberra", "egypt": "Cairo", "canada": "Ottawa",
        "germany": "Berlin", "india": "New Delhi",
    }
    result = capitals.get(country.lower().strip())
    if result:
        return f"The capital of {country} is {result}"
    return f"Capital not found for {country}"

print("Tools defined: calculator, word_count, lookup_capital")

## Example 1: Calculator Agent

An agent that uses the calculator tool to solve math problems.

In [ ]:
class MathQuestion(dspy.Signature):
    """Answer the math question using available tools."""
    question = dspy.InputField(desc="A math question")
    answer = dspy.OutputField(desc="The final numerical answer")

math_agent = dspy.ReAct(MathQuestion, tools=[calculator])

for q in ["What is 15 * 23 + 47?", "What is 1024 / 16?"]:
    try:
        result = math_agent(question=q)
        print_result(f"Q: {q}\nA: {result.answer}", "Calculator Agent")
    except Exception as e:
        print_error(f"Error on '{q}': {e}")

## Example 2: Multi-Tool Agent

An agent with access to multiple tools that decides which one to use.

In [ ]:
class GeneralQuestion(dspy.Signature):
    """Answer the question using available tools. Use tools when they can help."""
    question = dspy.InputField(desc="A general question")
    answer = dspy.OutputField(desc="The answer to the question")

multi_agent = dspy.ReAct(
    GeneralQuestion,
    tools=[calculator, word_count, lookup_capital],
    max_iters=5
)

for q in ["What is the capital of Japan?", "What is 256 * 3 + 42?"]:
    try:
        result = multi_agent(question=q)
        print_result(f"Q: {q}\nA: {result.answer}", "Multi-Tool Agent")
    except Exception as e:
        print_error(f"Error on '{q}': {e}")

## Example 3: Understanding the ReAct Loop

The ReAct pattern works in an iterative loop:
1. **THINK**: The agent reasons about what to do next
2. **ACT**: It chooses a tool and provides arguments
3. **OBSERVE**: It receives the tool's output
4. **REPEAT**: Back to thinking with new information
5. **FINISH**: Produce the final answer

In [ ]:
print("Example trace for 'What is the capital of France?':")
print("  Thought: I need to look up the capital of France")
print("  Action: lookup_capital('France')")
print("  Observation: The capital of France is Paris")
print("  Thought: I now know the answer")
print("  Answer: Paris")
print()
print("Writing good tool functions:")
print("  1. Clear function name (the agent uses it to decide which tool to call)")
print("  2. Descriptive docstring (the agent reads this to understand the tool)")
print("  3. Type-annotated parameters with descriptions")
print("  4. Return a string result")

## Example 4: When to Use ReAct

In [ ]:
print("Use ReAct when:")
print("  - The task requires external tools or data lookup")
print("  - The problem needs multiple steps with tool interactions")
print("  - You need the agent to decide which tools to use")
print("  - The task involves exploration or search")
print()
print("Use ChainOfThought instead when:")
print("  - No tools are needed (pure reasoning)")
print("  - The answer comes from the model's knowledge")
print("  - You want faster, cheaper responses")
print()
print("Key parameters for ReAct:")
print("  - tools: List of callable tool functions")
print("  - max_iters: Maximum reasoning-action cycles (default: 20)")

## Summary

**Key Takeaways:**
1. `ReAct` combines reasoning (Think) with acting (Tool Use)
2. Define tools as Python functions with clear docstrings
3. The agent decides which tool to call and with what arguments
4. Use `max_iters` to limit the number of reasoning-action cycles
5. Best for tasks that require external tools or multi-step interactions